# 03 · Real satellite data pipeline — international flaring (+ optional methane)

Builds the **real** flaring evidence GreenTruth verifies against, for major flaring
regions worldwide. No uploads and no login for the flaring data — it comes straight
from public World Bank links. Saves CSVs the app loads.

```
FLARING (primary, public, no login)
  World Bank Global Gas Flaring Tracker — July 2025 release, based on VIIRS,
  produced with the Colorado School of Mines / NOAA.
  Individual flare locations 2012–2024 (xlsx):
    https://thedocs.worldbank.org/en/doc/bd2432bbb0e514986f382f61b14b2608-0400072025/related/2012-2024-Flare-Volume-Estimates-by-individual-Flare-Location.xlsx
  Country totals 2012–2024 (xlsx):
    https://thedocs.worldbank.org/en/doc/bd2432bbb0e514986f382f61b14b2608-0400072025/related/Flare-volume-and-intensity-estimates-2012-2024.xlsx
  Landing page: https://www.worldbank.org/en/programs/gasflaringreduction/global-flaring-data
  Classified Public by the World Bank. Credit: World Bank GFMR + Earth Observation
  Group, Payne Institute, Colorado School of Mines.

METHANE (optional cross-check, needs a free Google Earth Engine account)
  Sentinel-5P OFFL CH4 (TROPOMI) on Earth Engine — COPERNICUS/S5P/OFFL/L3_CH4.
  ~7 km, 2019–present. Coarse: a regional cross-check, not a per-facility measurement.

CLAIMS (real, public): the World Bank "Zero Routine Flaring by 2030" initiative
  https://www.worldbank.org/en/programs/gasflaringreduction/zero-routine-flaring-by-2030
```

**Honesty note.** Everything saved here is real, labelled with its source. There is
no synthetic data in this project. A flare near a field is consistent with activity
there, not proof any operator caused it — keep every verdict worded that way.

Accessed: 17 Sep 2026.

| Notebook card | |
|---|---|
| **Type** | Process real data |
| **Purpose** | Download the public World Bank / VIIRS flaring releases, group flares into the 12 monitored fields, and (Part B, optional) extract Sentinel-5P methane. |
| **Inputs** | Public World Bank .xlsx releases (no account). Part B: Google Earth Engine (free account + Cloud project id). |
| **Outputs** | `flaring_by_field.csv`, `flaring_by_country.csv`, `flaring_by_operator.csv`, `methane_by_field_s5p.csv` → place in `data/real/`. |
| **Where it runs** | Google Colab or locally. |
| **Execution record** | Executed in Colab — record in `notebooks/executed/03_real_satellite_data_pipeline.ipynb`; outputs in `notebooks/executed/results/` and `data/real/` (not committed: external licence). |


## Part A — international flaring volumes (World Bank / VIIRS)

In [ ]:
!pip install -q pandas numpy openpyxl requests matplotlib

In [ ]:
import requests, pandas as pd, numpy as np, math, os

URL_SITES = ("https://thedocs.worldbank.org/en/doc/"
             "bd2432bbb0e514986f382f61b14b2608-0400072025/related/"
             "2012-2024-Flare-Volume-Estimates-by-individual-Flare-Location.xlsx")
URL_COUNTRY = ("https://thedocs.worldbank.org/en/doc/"
               "bd2432bbb0e514986f382f61b14b2608-0400072025/related/"
               "Flare-volume-and-intensity-estimates-2012-2024.xlsx")

def download(url, path):
    r = requests.get(url, timeout=180); r.raise_for_status()
    open(path, "wb").write(r.content)
    print("downloaded", path, f"({len(r.content)//1024} KB)"); return path

download(URL_SITES, "wb_sites.xlsx")
download(URL_COUNTRY, "wb_country.xlsx")

### A1 · The fields we group flares into (inlined — nothing to upload)

In [ ]:
FIELDS = {
    "us_permian":        {"name": "Permian Basin", "country": "United States", "lat": 31.9,  "lon": -102.1, "match_radius_km": 200},
    "us_bakken":         {"name": "Bakken",         "country": "United States", "lat": 47.8,  "lon": -103.3, "match_radius_km": 150},
    "russia_priobskoye": {"name": "Priobskoye / West Siberia", "country": "Russia", "lat": 60.9, "lon": 69.6, "match_radius_km": 150},
    "iraq_rumaila":      {"name": "Rumaila / Basra", "country": "Iraq",    "lat": 30.05, "lon": 47.3,  "match_radius_km": 90},
    "iran_south_pars":   {"name": "South Pars / Asaluyeh", "country": "Iran", "lat": 27.5, "lon": 52.6, "match_radius_km": 90},
    "algeria_hassi_messaoud": {"name": "Hassi Messaoud", "country": "Algeria", "lat": 31.67, "lon": 6.07, "match_radius_km": 60},
    "algeria_hassi_rmel":     {"name": "Hassi R\'Mel", "country": "Algeria", "lat": 32.93, "lon": 3.28, "match_radius_km": 60},
    "nigeria_niger_delta": {"name": "Niger Delta", "country": "Nigeria", "lat": 5.3, "lon": 6.0, "match_radius_km": 180},
    "venezuela_maracaibo": {"name": "Lake Maracaibo", "country": "Venezuela", "lat": 9.8, "lon": -71.5, "match_radius_km": 150},
    "libya_sirte":       {"name": "Sirte Basin", "country": "Libya", "lat": 28.8, "lon": 19.6, "match_radius_km": 180},
    "kazakhstan_tengiz": {"name": "Tengiz", "country": "Kazakhstan", "lat": 46.1, "lon": 53.5, "match_radius_km": 80},
    "mexico_cantarell":  {"name": "Cantarell / Campeche", "country": "Mexico", "lat": 19.4, "lon": -92.2, "match_radius_km": 120},
}
def haversine_km(a, b, c, d):
    R = 6371.0; p1, p2 = math.radians(a), math.radians(c)
    dphi, dl = math.radians(c - a), math.radians(d - b)
    x = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(x))

def nearest_field(lat, lon):
    best, bestd, best_r = None, 1e9, 0
    for fid, f in FIELDS.items():
        dd = haversine_km(lat, lon, f["lat"], f["lon"])
        if dd < bestd: best, bestd, best_r = fid, dd, f["match_radius_km"]
    return best if bestd <= best_r else None

### A2 · Load the site-level sheet (auto-detect the header row and columns)

World Bank sheets sometimes have a title row, so we find the header, then detect the
latitude/longitude columns and the per-year volume columns.

In [ ]:
def read_excel_smart(path):
    raw = pd.read_excel(path, header=None, engine="openpyxl")
    hrow = 0
    for i in range(min(12, len(raw))):
        vals = [str(x).lower() for x in raw.iloc[i].tolist()]
        if any("latitude" in v for v in vals) or any(v == "lat" for v in vals):
            hrow = i; break
    df = pd.read_excel(path, header=hrow, engine="openpyxl")
    df.columns = [" ".join(str(c).split()) for c in df.columns]   # collapse double spaces
    return df

sites = read_excel_smart("wb_sites.xlsx")
print("columns:", list(sites.columns))

def find(cols, *keys):
    for c in cols:
        cl = c.lower()
        if any(k in cl for k in keys): return c
    return None

# The World Bank site file is LONG format: one row per flare per year.
latc = find(sites.columns, "latitude", "lat")
lonc = find(sites.columns, "longitude", "lon")
yearc = find(sites.columns, "year")
volc = find(sites.columns, "bcm")                 # billion m3
vol_is_million = False
if volc is None:                                  # fall back to the million-m3 column
    volc = find(sites.columns, "million m3", "flaring vol"); vol_is_million = True
namec = find(sites.columns, "field name")
opc = find(sites.columns, "operator")
ctyc = find(sites.columns, "country")
print("lat:", latc, "| lon:", lonc, "| year:", yearc, "| vol:", volc,
      "| field:", namec, "| operator:", opc)
assert latc and lonc and yearc and volc, "Adjust detection to the printed columns."
sites.head()

### A3 · Match flares to fields and build the real per-field annual series

In [ ]:
def to_bcm(v):
    v = float(v)
    return v / 1000.0 if vol_is_million else v      # million m3 -> billion m3

rows = []
for _, r in sites.iterrows():
    try:
        la, lo = float(r[latc]), float(r[lonc])
        yr, vol = int(r[yearc]), to_bcm(r[volc])
    except (TypeError, ValueError):
        continue
    fid = nearest_field(la, lo)
    if fid:
        rows.append((fid, yr, vol))

by_field = (pd.DataFrame(rows, columns=["field", "year", "volume"])
            .groupby(["field", "year"], as_index=False)["volume"].sum())
by_field["source"] = "World Bank Global Gas Flaring Tracker (VIIRS) — REAL"
by_field.to_csv("flaring_by_field.csv", index=False)
print("saved flaring_by_field.csv:", by_field["field"].nunique(), "fields,",
      len(by_field), "rows")
by_field.head(20)

# BONUS: the file already names the real field and operator, so we can also build
# operator-level series straight from the source (no coordinate matching).
if opc or namec:
    key = opc or namec
    op = sites[[key, yearc, volc]].copy()
    op["volume"] = op[volc].apply(to_bcm)
    op["year"] = pd.to_numeric(op[yearc], errors="coerce")
    op = (op.dropna(subset=["year", key])
            .groupby([key, "year"], as_index=False)["volume"].sum()
            .rename(columns={key: "operator"}))
    op["operator"] = op["operator"].astype(str)
    op.to_csv("flaring_by_operator.csv", index=False)
    print("saved flaring_by_operator.csv:", op["operator"].nunique(), "operators")

### A4 · Country totals (a clean international overview)

In [ ]:
country = read_excel_smart("wb_country.xlsx")
print("columns:", list(country.columns)[:20])
cc = find(country.columns, "country")
ycols = [c for c in country.columns if str(c)[:4].isdigit() and 2012 <= int(str(c)[:4]) <= 2024]
if cc and ycols:
    long = country.melt(id_vars=[cc], value_vars=ycols, var_name="year", value_name="volume")
    long = long.rename(columns={cc: "country"}).dropna(subset=["volume"])
    long["year"] = long["year"].astype(str).str[:4].astype(int)
    long["source"] = "World Bank Global Gas Flaring Tracker — REAL"
    long.to_csv("flaring_by_country.csv", index=False)
    print("saved flaring_by_country.csv:", long["country"].nunique(), "countries")
    long.head()
else:
    print("Check the country sheet columns and set cc / ycols by hand.")

## Part B — Sentinel-5P methane cross-check (optional; free GEE account)

In [ ]:
BAND = "CH4_column_volume_mixing_ratio_dry_air"
GEE_PROJECT = ""   # <-- put your free Earth Engine Cloud project id here, e.g. "ee-yourname"
assert GEE_PROJECT, ("Set GEE_PROJECT to your free GEE project id "
                     "(https://code.earthengine.google.com/ -> project id). "
                     "Part B is optional; skip it if you don't need methane.")
ee.Initialize(project=GEE_PROJECT)
col = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_CH4").select(BAND)

In [ ]:
BAND = "CH4_column_volume_mixing_ratio_dry_air"
col = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_CH4").select(BAND)
rows = []
for fid, f in FIELDS.items():
    region = ee.Geometry.Point([f["lon"], f["lat"]]).buffer(25000)
    for year in range(2019, 2025):
        for month in range(1, 13):
            start = ee.Date.fromYMD(year, month, 1); end = start.advance(1, "month")
            try:
                v = col.filterDate(start, end).mean().reduceRegion(
                    ee.Reducer.mean(), region, scale=1113).get(BAND).getInfo()
            except Exception:
                v = None
            if v is not None:
                rows.append({"field": fid, "year": year, "month": month, "ch4_ppb": v})
    print("done", fid)
ch4 = pd.DataFrame(rows); ch4["source"] = "Sentinel-5P OFFL CH4 (Copernicus/ESA via GEE) — REAL"
ch4.to_csv("methane_by_field_s5p.csv", index=False)
print("saved methane_by_field_s5p.csv:", len(ch4), "monthly points")

Sentinel-5P is ~7 km, so a value is a **regional** methane column, not one facility (land only before Nov 2021; gap 26 Jul–31 Aug 2022; double the reported error). That is why methane is a cross-check, not primary evidence.

## Part C · Quick look

In [ ]:
import matplotlib.pyplot as plt
for fid, g in by_field.groupby("field"):
    g = g.sort_values("year")
    plt.plot(g["year"], g["volume"], marker="o", label=FIELDS[fid]["name"])
plt.xlabel("year"); plt.ylabel("flared volume (bcm)")
plt.title("Real flaring by field (World Bank / VIIRS)"); plt.legend(fontsize=7, ncol=2)
plt.show()

## Outputs → the app

Download from the Colab Files panel into `data/real/`:
- `flaring_by_field.csv` — real per-field annual volumes (the app loads this).
- `flaring_by_country.csv` — real country totals (international overview).
- `methane_by_field_s5p.csv` — optional real methane cross-check.

The field ids match `data/facilities_international.json`, so the app picks them up
directly. Verdicts compare a public commitment (e.g. Zero Routine Flaring by 2030)
against the real observed trajectory, worded as consistency with observations.